# Homework 10 — Kubernetes and Model Serving — ML Zoomcamp 2026 cohort

Deploy the **HW5 lead-scoring API** (FastAPI, `05-deployment/`) to a named local `kind` cluster: one Deployment, one `ClusterIP` Service reached with `kubectl port-forward`, one HPA. No TF Serving, no gateway. Questions: `homework10_2026.md`.

**Submit at:** https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw10 · platform deadline **Mon 14 Dec**. Target: submit on the Mon 7 Dec 08:00 session. The form also asks for a public link to your work and a short reflection ("one practical idea you took from this module").

This is terminal work. Run the commands in a terminal **from this `homework/` folder** and use this notebook as the checklist: note what you observed in the empty cell under each question. Scaffold only: no answers.

> `homework10_2026.md` is written for the course repo layout (`cd cohorts/2026/homework/05-deployment`, `python ../05-deployment/q6_test.py`). Here the HW5 release is the subfolder `05-deployment/`, so the paths below differ from the official text.

## Setup — files and model checksum

In [ ]:
from pathlib import Path
import hashlib

HW5 = Path('05-deployment')

# WHY: the Dockerfile COPYs these; a missing one (often the dotfile .python-version) fails the build.
needed = ['Dockerfile', 'pyproject.toml', 'uv.lock', '.python-version', 'model.py', 'predict.py',
          'feature_defaults.json', 'model_metadata.json', 'pipeline.bin', 'q6_test.py']
missing = [f for f in needed if not (HW5 / f).exists()]
print('missing:', missing or 'none')

# The checksum the HW5 instructions give for the frozen artifact.
expected = '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1'
print('pipeline.bin checksum ok:', hashlib.sha256((HW5 / 'pipeline.bin').read_bytes()).hexdigest() == expected)

print('manifests here:', sorted(p.name for p in Path('.').glob('*.yaml')))

## Q1. Local container — `conversion_probability`

Docker Desktop must be running. Build the image with the **HW10 tag** and run it:

```bash
cd 05-deployment
docker build -t zoomcamp-model:2026-hw10 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw10
```

Second terminal, from `homework/`:

```bash
python 05-deployment/q6_test.py
```

Report `conversion_probability` rounded to **3 decimals** (tolerance 0.005; there are no options to pick from).

> The tag must be exactly `zoomcamp-model:2026-hw10`: `deployment.yaml` refers to it. If you built `zoomcamp-model:2026-hw5` in Module 5, `docker tag zoomcamp-model:2026-hw5 zoomcamp-model:2026-hw10` gives the same image a second name.
> Stop this container (Ctrl-C) before Q7: the port-forward needs local port 9696.

## Q2. Environment check (not graded)

```bash
kind --version
kubectl version --client
```

Record the output. No answer to submit.

## Create the named cluster

```bash
kind create cluster --name mlzoomcamp-2026
kubectl cluster-info --context kind-mlzoomcamp-2026
```

> Two names for one cluster: `kind ... --name mlzoomcamp-2026`, but `kubectl ... --context kind-mlzoomcamp-2026`. Keep the `--context` flag on every `kubectl` command: if Docker Desktop's own Kubernetes or another kind cluster exists, `kubectl` may otherwise talk to that one.

## Q3. Kubernetes primitives

The smallest deployable computing unit Kubernetes creates and manages. Lesson 10.5. *(Node / Pod / Deployment / Service)*

## Q4. Default service type

```bash
kubectl get services --context kind-mlzoomcamp-2026
```

`TYPE` of the service named `kubernetes`. Read it from the output, not from memory. *(NodePort / ClusterIP / ExternalName / LoadBalancer)*

## Q5. Load the local image into the cluster

```bash
kind load docker-image zoomcamp-model:2026-hw10 --name mlzoomcamp-2026
```

Which command performs this operation? *(kind create cluster / kind build node-image / kind load docker-image / kubectl apply)*

> Without `--name`, kind looks for a cluster called `kind` and fails with "no nodes found". Load again after every rebuild or new tag.

## Q6. Deploy the API

```bash
kubectl apply -f deployment.yaml --context kind-mlzoomcamp-2026
kubectl rollout status deployment/subscription --context kind-mlzoomcamp-2026
kubectl get pods --context kind-mlzoomcamp-2026
```

Which container port does `deployment.yaml` use? Open the file. *(80 / 8080 / 9000 / 9696)*

> `ErrImagePull` / `ImagePullBackOff` = the cluster does not have that exact image tag (Q5 skipped, wrong `--name`, or a different tag). `kubectl describe pod <name> --context kind-mlzoomcamp-2026` shows the reason under *Events*. `READY 0/1` for a few seconds is normal: the readiness probe on `/health` has not passed yet.

## Q7. Expose the deployment

```bash
kubectl apply -f service.yaml --context kind-mlzoomcamp-2026
kubectl get service subscription --context kind-mlzoomcamp-2026
```

Which selector value routes traffic to the deployment? Open `service.yaml`. *(app: api / app: subscription / app: lead-scoring / app: zoomcamp-model)*

Then test through the service (port-forward blocks: keep it in its own terminal):

```bash
kubectl port-forward service/subscription 9696:80 --context kind-mlzoomcamp-2026
python 05-deployment/q6_test.py          # second terminal, from homework/
```

The probability must match Q1 within 0.005.

> `9696:80` = your Mac's 9696 → the **service** port 80 → the pod's `targetPort`. Writing `9696:9696` fails, because the service does not expose 9696. "address already in use" = the Q1 container is still running.

## Autoscaling (not graded)

```bash
kubectl apply -f hpa.yaml --context kind-mlzoomcamp-2026
kubectl get hpa subscription-hpa --context kind-mlzoomcamp-2026
```

> `TARGETS <unknown>` is expected on a fresh kind cluster: there is no metrics-server. Optional: install it with the two commands in the workshop section of `../practice/10-kubernetes_commented.ipynb` (add `--context kind-mlzoomcamp-2026`), then watch `kubectl get hpa subscription-hpa -w` while a loop posts requests. The observed replica count is not graded.

## Q8. HPA configuration

`maxReplicas` declared in `hpa.yaml`: read the file, not the live replica count. *(1 / 2 / 3 / 4)*

## Clean up

```bash
kind delete cluster --name mlzoomcamp-2026
```

Then submit on the form: the answers it asks for (Q1 with 3 decimals), the link to your work, the reflection.